In [ ]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import math
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from gensim.models import Word2Vec
import warnings

warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Web Scraper and Dataset Preparation

def scrape_hindi_news(url):

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.content, 'html.parser')

        paragraphs = soup.find_all('p')
        text = " ".join([p.get_text() for p in paragraphs if len(p.get_text()) > 20])
        return text
    except Exception as e:
        print(f"Scraping failed for {url}: {e}")
        return None

dataset_path = '/kaggle/input/datasets/arnavagrawal22/arnsin-dl-cleaned1/dataset-merged.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset loaded. Total records: {len(df)}")

Dataset loaded. Total records: 17124


In [ ]:
#Custom Tokenizer and Stopword Logic

import re

# Creating our own stopword list instead of relying on NLTK
hindi_stopwords = set([
    "है", "और", "कि", "का", "की", "के", "में", "से", "को", "पर", 
    "यह", "वह", "जो", "तो", "भी", "ने", "एक", "हो", "कर", "साथ"
])

def custom_hindi_tokenizer(text):
    
    text = str(text) 
    
    text = re.sub(r'[!\"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~।]', ' ', text)
    
    tokens = text.split()
    
    # Remove stopwords
    filtered_tokens = [word for word in tokens if word not in hindi_stopwords]
    return filtered_tokens

# Drop rows where 'text' is completely missing (NaN)
df = df.dropna(subset=['text'])

df['text'] = df['text'].astype(str)

df['tokens'] = df['text'].apply(custom_hindi_tokenizer)
print("Sample tokens:", df['tokens'].iloc[0])

Sample tokens: ['‘मोदी', 'शासन', 'दौरान', 'गंगा’', 'गंगा', 'नदी', 'नरेन्द्र', 'मोदी', 'शाशन', 'कांग्रेस', 'मुकाबले', 'काफ़ी', 'स्वच्छ']


In [ ]:
#Fundamental Math Models Built from Scratch

# --- 1. TF-IDF ---
def build_custom_tfidf(tokenized_docs):
    doc_count = len(tokenized_docs)
    tf = []
    df_counts = defaultdict(int)
    
    # Calculate TF
    for doc in tokenized_docs:
        doc_tf = defaultdict(float)
        word_count = len(doc)
        for word in doc:
            doc_tf[word] += 1 / word_count if word_count > 0 else 0
        tf.append(doc_tf)
        
        # Document Frequency
        for word in set(doc):
            df_counts[word] += 1
            
    # Calculate IDF and final TF-IDF
    tfidf_matrix = []
    vocabulary = list(df_counts.keys())
    
    for doc_tf in tf:
        doc_tfidf = []
        for word in vocabulary:
            if word in doc_tf:
                idf = math.log(doc_count / (df_counts[word] + 1))
                doc_tfidf.append(doc_tf[word] * idf)
            else:
                doc_tfidf.append(0.0)
        tfidf_matrix.append(doc_tfidf)
        
    return np.array(tfidf_matrix), vocabulary

X_tfidf, vocab = build_custom_tfidf(df['tokens'].tolist())
y = df['label'].values

# --- 2. Scratch Naive Bayes ---
class CustomNaiveBayes:
    def fit(self, X, y):
        self.num_classes = len(np.unique(y))
        self.priors = np.zeros(self.num_classes)
        self.means = np.zeros((self.num_classes, X.shape[1]))
        self.vars = np.zeros((self.num_classes, X.shape[1]))
        
        for c in range(self.num_classes):
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / X.shape[0]
            self.means[c, :] = X_c.mean(axis=0)
            self.vars[c, :] = X_c.var(axis=0) + 1e-9 # Added small epsilon to prevent division by zero

    def _calculate_likelihood(self, class_idx, x):
        mean = self.means[class_idx]
        var = self.vars[class_idx]
        numerator = np.exp(- (x - mean)**2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return np.sum(np.log(numerator / denominator)) # Used log probabilities to prevent underflow

    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for c in range(self.num_classes):
                prior = np.log(self.priors[c])
                likelihood = self._calculate_likelihood(c, x)
                posteriors.append(prior + likelihood)
            predictions.append(np.argmax(posteriors))
        return np.array(predictions)

# Train Baseline
nb_model = CustomNaiveBayes()
nb_model.fit(X_tfidf, y)
baseline_preds = nb_model.predict(X_tfidf)
baseline_acc = np.mean(baseline_preds == y)
print(f"Scratch Naive Bayes Accuracy (on training data): {baseline_acc * 100:.2f}%")

Scratch Naive Bayes Accuracy (on training data): 59.58%


In [ ]:
# ---1. Train Custom Word Vectors ---
w2v_model = Word2Vec(sentences=df['tokens'].tolist(), vector_size=50, window=5, min_count=1, workers=4)

def text_to_sequence(tokens, w2v_model, max_len=20):
    seq = []
    for word in tokens:
        if word in w2v_model.wv:
            seq.append(w2v_model.wv[word])
    
    # Pad or truncate sequence
    if len(seq) < max_len:
        padding = [np.zeros(50)] * (max_len - len(seq))
        seq.extend(padding)
    else:
        seq = seq[:max_len]
    return np.array(seq)

# Prepare Deep Learning Data
X_dl = np.array([text_to_sequence(tokens, w2v_model) for tokens in df['tokens']])
y_dl = torch.tensor(df['label'].values, dtype=torch.float32).view(-1, 1)
X_dl = torch.tensor(X_dl, dtype=torch.float32)

class HindiNewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

dataloader = DataLoader(HindiNewsDataset(X_dl, y_dl), batch_size=2, shuffle=True)

# --- 2. PyTorch LSTM Architecture ---
class FakeNewsLSTM(nn.Module):
    def __init__(self, input_size=50, hidden_size=64):
        super(FakeNewsLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :] # Take the output of the last time step
        out = self.fc(out)
        return self.sigmoid(out)

lstm_model = FakeNewsLSTM().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.01)

# Training Loop
print("Training PyTorch LSTM on GPU...")
epochs = 20
for epoch in range(epochs):
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("LSTM Training Complete.")

Training PyTorch LSTM on GPU...
Epoch [5/20], Loss: 0.2311
Epoch [10/20], Loss: 0.2138
Epoch [15/20], Loss: 0.5045
Epoch [20/20], Loss: 1.0799
LSTM Training Complete.


In [ ]:
# 1. Saving Vectorizer Vocabulary (for our custom TF-IDF)
np.save('custom_vocab.npy', np.array(vocab))

# 2. Saving Baseline Model Parameters
np.save('nb_means.npy', nb_model.means)
np.save('nb_vars.npy', nb_model.vars)
np.save('nb_priors.npy', nb_model.priors)

# 3. Saving Deep Learning Models
w2v_model.save("hindi_word2vec.model")
torch.save(lstm_model.state_dict(), 'lstm_fake_news.pth')

print("All components saved successfully! You can download these from the Kaggle 'Output' directory.")

All components saved successfully! You can download these from the Kaggle 'Output' directory.
